# CfD settlement trace
Recalculating CfD generator payments for Settlement Day 12 April 2026 from public Elexon and LCCC data.

In [ ]:
import duckdb
import pandas as pd

## 1. Check every unit (SQL)
If every MWh earned a Difference Amount, (strike - weighted price) x generation equals the payment.

In [ ]:
duckdb.sql("""
SELECT contract_round,
       count(*)                                           AS units,
       count(*) FILTER (WHERE unpaid)                     AS units_with_unpaid_hours,
       round(sum(generation_mwh))                         AS generation_mwh,
       round(sum(CASE WHEN unpaid THEN gap / strike ELSE 0 END)) AS unpaid_mwh
FROM (
    SELECT *,
           expected - payment_gbp AS gap,
           abs(expected - payment_gbp) > greatest(1, 0.0005 * abs(expected)) AS unpaid
    FROM (
        SELECT *, (strike - weighted_price) * generation_mwh AS expected
        FROM 'data/cfd_actuals_2026-04-12.csv'
        WHERE reference_type = 'IMRP' AND generation_mwh > 0
    )
)
GROUP BY contract_round
ORDER BY contract_round
""")

## 2. Recalculate three payments hour by hour (Python)
Results are before loss adjustment.

In [ ]:
imrp  = pd.read_csv("data/imrp_2026-04-12.csv")    # hour, price
b1610 = pd.read_csv("data/b1610_2026-04-12.csv")   # bm_unit, settlement_period, mwh

def difference_amount(bm_unit, strike, rule):
    hh = b1610[b1610.bm_unit == bm_unit].copy()
    hh["hour"] = (hh.settlement_period + 1) // 2          # two half-hours per hourly price
    hh["mwh"]  = hh.mwh.clip(lower=0)                     # ignore small imports
    df = hh.groupby("hour", as_index=False).mwh.sum().merge(imrp, on="hour")

    negative = df.price < 0
    run_length = negative.groupby((negative != negative.shift()).cumsum()).transform("sum")
    paid = {"floor": pd.Series(True, index=df.index),      # negative price treated as zero
            "six":   ~(negative & (run_length >= 6)),      # no payment in a run of 6+ negative hours
            "any":   ~negative}[rule]                      # no payment in any negative hour

    return (df.mwh * (strike - df.price.clip(lower=0)) * paid).sum()

for name, bm_unit, strike, rule, published in [
    ("Cleve Hill",          "T_CLVHS-1",  62.99, "any",   3606.45),
    ("Hornsea Two Phase 1", "T_HOWBO-1",  82.49, "six", 116393.73),
    ("Hornsea One Phase 1", "T_HOWAO-2", 206.06, "six", 484233.18)]:
    mine = difference_amount(bm_unit, strike, rule)
    print(f"{name:20s} mine {mine:12,.2f}  published {published:12,.2f}  {100 * (mine / published - 1):+.2f}%")

## 3. Boundary test: 11 April, price below zero for five hours
Five hours is one short of the six-hour rule.

In [ ]:
duckdb.sql("""
SELECT contract_round,
       count(*) AS units_checked,
       count(*) FILTER (WHERE abs((strike - weighted_price) * generation_mwh - payment_gbp)
                              <= greatest(1, 0.0005 * abs((strike - weighted_price) * generation_mwh))) AS paid_on_every_mwh
FROM 'data/cfd_actuals_2026-04-11_sample.csv'
GROUP BY contract_round
ORDER BY contract_round
""")